In [ ]:
# ============================================
# CELL 0 — Install deps (run once per runtime)
# ============================================
!pip install -q pandas numpy pyarrow fastparquet spacy tqdm ftfy flashtext

#!python -m spacy download en_core_web_sm -q



  Preparing metadata (setup.py) ... done


In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 2) Imports
import os, re, json, html, hashlib, sys, subprocess
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from flashtext import KeywordProcessor

# ---------------- Paths (EDIT BASE if needed) ----------------
# Accept misspelt "PropInisght", fallback to "PropInsight"
BASE = Path("/content/drive/MyDrive/PropInsight")

RAW_DIR      = BASE / "raw" / "common_reddit"
CORPUS_DIR   = BASE / "corpus"
SINGLISH_DIR = CORPUS_DIR / "Singlish"         # dictionary/, vocabulary/, lexicon.csv
PROP_DIR     = CORPUS_DIR / "SGPropertyDomain" # glossary.csv, regex_patterns.jsonl, etc.
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Input CSV: newest reddit_corpus_2023_2025*.csv
candidates = sorted(RAW_DIR.glob("reddit_corpus_2023_2025*.csv"))
if not candidates:
    raise FileNotFoundError(f"No reddit_corpus_2023_2025*.csv found in {RAW_DIR}")
INPUT_FILE = candidates[-1]

# Outputs (same folder, suffix _common)
OUTPUT_FILE = RAW_DIR / "reddit_corpus_enhanced_common.csv"
REPORT_FILE = RAW_DIR / "reddit_preprocessing_report_common.json"
NLP_OUT     = RAW_DIR / "reddit_corpus_enhanced+nlp_common.parquet"

# EntityRuler patterns (optional)
_ruler_candidates = (
    list(PROP_DIR.glob("spacy_entityruler_patterns.merged.jsonl")) +
    list(PROP_DIR.glob("spacy_entityruler_patterns.jsonl")) +
    list(PROP_DIR.glob("spacy_entityruler_pattern*.jsonl"))
)
RULER_PATH = _ruler_candidates[0] if _ruler_candidates else None

print("✅ Using paths")
print("Base      :", BASE)
print("Input     :", INPUT_FILE)
print("Output    :", OUTPUT_FILE)
print("Report    :", REPORT_FILE)
print("NLP out   :", NLP_OUT)
print("Singlish  :", SINGLISH_DIR)
print("Property  :", PROP_DIR)
print("Ruler     :", RULER_PATH if RULER_PATH else "(none found)")

# ---------------- Core regexes ----------------
URL_PATTERN      = re.compile(r"https?://\S+|www\.\S+", re.I)
MD_LINK_PATTERN  = re.compile(r"\[([^\]]+)\]\((https?:\/\/[^\)]+)\)")
HTML_TAG_PATTERN = re.compile(r"<[^>]+>")

# ---------------- Robust CSV reader ----------------
def read_csv_flex(path: Path):
    trials = [
        dict(encoding="utf-8"),
        dict(encoding="utf-8-sig"),
        dict(encoding="latin1"),
        dict(engine="python"),
    ]
    for kw in trials:
        try:
            return pd.read_csv(path, **kw)
        except Exception:
            pass
    try:
        return pd.read_csv(path, sep="\t", engine="python")
    except Exception:
        return None

# ---------------- Corpus loaders ----------------
class SinglishCorpusLoader:
    """Recursively loads Singlish terms from CSV/TXT under /corpus/Singlish."""
    def __init__(self, corpus_path: Path):
        self.corpus_path    = Path(corpus_path)
        self.singlish_terms = set()
        self.load()

    def _csv_first_col(self, df: pd.DataFrame):
        cols = {c.lower(): c for c in df.columns}
        for key in ["word", "term", "singlish", "lexeme"]:
            if key in cols:
                return cols[key]
        return list(df.columns)[0]

    def load(self):
        found = 0
        for fp in sorted(self.corpus_path.rglob("*.csv")):
            df = read_csv_flex(fp)
            if df is None or df.empty:
                continue
            col = self._csv_first_col(df)
            for val in df[col].dropna():
                term = str(val).strip().lower()
                if term:
                    self.singlish_terms.add(term); found += 1
        for fp in sorted(self.corpus_path.rglob("*.txt")):
            try:
                for ln in fp.read_text(encoding="utf-8", errors="ignore").splitlines():
                    term = ln.strip().lower()
                    if term:
                        self.singlish_terms.add(term); found += 1
            except Exception:
                pass
        print(f"📘 Singlish terms loaded: {len(self.singlish_terms)} (from {found} entries)")

class PropertyDomainCorpus:
    """Loads property terms from glossary.csv (or glossary*.csv) + any *.txt under SGPropertyDomain."""
    def __init__(self, corpus_path: Path):
        self.corpus_path = Path(corpus_path)
        self.terms = {}
        self.load()

    def _pick_glossary(self):
        primary = self.corpus_path / "glossary.csv"
        if primary.exists(): return primary
        fallbacks = sorted(self.corpus_path.glob("glossary*.csv"))
        return fallbacks[-1] if fallbacks else None

    def load(self):
        count = 0
        gl = self._pick_glossary()
        if gl and gl.exists():
            df = read_csv_flex(gl)
            if df is not None and not df.empty:
                cols = {c.lower(): c for c in df.columns}
                term_col  = cols.get("term", list(df.columns)[0])
                cat_col   = cols.get("category", None)
                alias_col = cols.get("aliases", None)
                for _, r in df.iterrows():
                    term = str(r.get(term_col, "")).strip().lower()
                    if not term: continue
                    cat  = str(r.get(cat_col, "general")) if cat_col else "general"
                    self.terms[term] = cat; count += 1
                    if alias_col and pd.notna(r.get(alias_col)):
                        for a in str(r.get(alias_col)).split("|"):
                            alias = a.strip().lower()
                            if alias:
                                self.terms[alias] = cat; count += 1
            print(f"📗 Property terms loaded from {gl.name if gl else '(none)'}: {len(self.terms)}")
        for fp in sorted(self.corpus_path.glob("*.txt")):
            try:
                for ln in fp.read_text(encoding="utf-8", errors="ignore").splitlines():
                    term = ln.strip().lower()
                    if term:
                        self.terms[term] = "general"; count += 1
            except Exception:
                pass
        if count == 0 and not self.terms:
            print("⚠️ No property glossary/terms found.")

# ---------------- Preprocessor (vectorized) ----------------
processor_sing = SinglishCorpusLoader(SINGLISH_DIR)
processor_prop = PropertyDomainCorpus(PROP_DIR)

# Build FlashText keyword processors (linear-time matching)
sing_kp = KeywordProcessor(case_sensitive=False)
for w in sorted(processor_sing.singlish_terms):
    sing_kp.add_keyword(w)

prop_kp = KeywordProcessor(case_sensitive=False)
for w in sorted(processor_prop.terms.keys()):
    prop_kp.add_keyword(w)

# Load CSV
df = read_csv_flex(INPUT_FILE)
if df is None or df.empty:
    raise ValueError(f"Failed to read data from {INPUT_FILE}")
print(f"📦 Loaded {len(df)} rows")

# Ensure text cols exist
text_cols = ["title","selftext","comment_body","body"]
for c in text_cols:
    if c not in df.columns: df[c] = ""

# Vectorized cleaning
def _clean_series(s):
    s = s.astype(str).map(html.unescape)
    s = s.str.replace(r"https?://\S+|www\.\S+", " ", regex=True)
    s = s.str.replace(r"\[([^\]]+)\]\((https?:\/\/[^\)]+)\)", r"\1", regex=True)
    s = s.str.replace(r"<[^>]+>", " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

df["body_clean"] = _clean_series(df[text_cols].fillna("").agg(" ".join, axis=1))
df["content_hash"] = df["body_clean"].map(lambda x: hashlib.md5(x.encode()).hexdigest())
df["wc"]   = df["body_clean"].str.split().map(len)
df["uniq"] = df["body_clean"].str.split().map(lambda xs: 0 if not xs else len(set(xs))/len(xs))
df["quality_score"] = (df["uniq"].clip(0,1)*0.5 + (df["wc"]/50).clip(0,1)*0.5)
df["is_spam"] = df["wc"] < 5

# Early dedupe & spam filter
before = len(df)
df = (df[~df["is_spam"]]
      .sort_values("quality_score", ascending=False)
      .drop_duplicates("content_hash")
      .copy())
after = len(df)
print(f"🧹 Filtered spam/dupes: {before-after} rows removed; {after} remain")

# Fast dictionary hits
def _hits(text):
    s = " ".join(text.split())
    sing = sing_kp.extract_keywords(s)
    prop = prop_kp.extract_keywords(s)
    return "|".join(sorted(set(sing))), "|".join(sorted(set(prop)))

pairs = df["body_clean"].map(_hits)
df["singlish_terms"]  = [a for a,b in pairs]
df["property_terms"]  = [b for a,b in pairs]

# Save quick CSV + report
df_out = df.drop(columns=["wc","uniq"], errors="ignore").copy()
df_out.to_csv(OUTPUT_FILE, index=False)

report = dict(
    records=len(df_out),
    avg_quality=float(df_out["quality_score"].mean()) if len(df_out) else 0.0,
    singlish_hits=int((df_out["singlish_terms"].astype(str)!="").sum()),
    property_hits=int((df_out["property_terms"].astype(str)!="").sum())
)
with open(REPORT_FILE,"w") as f: json.dump(report,f,indent=2)
print("📄 Report:", report)
# ---------------- spaCy (tokens / lemmas / noun_phrases) ----------------
import spacy
MODEL = "en_core_web_sm"
try:
    nlp = spacy.load(MODEL, disable=["ner"])  # keep tagger+parser+lemmatizer for chunks & lemmas
except OSError:
    subprocess.run([sys.executable, "-m", "spacy", "download", MODEL], check=True)
    nlp = spacy.load(MODEL, disable=["ner"])

# (Note: EntityRuler would affect NER; we're disabling NER for speed in this pass.)
nlp.max_length = 2_000_000
BATCH_SIZE = 256
N_PROC = max(1, os.cpu_count() - 1)
print(f"⚙️ spaCy tokens/lemmas/chunks with n_process={N_PROC}, batch_size={BATCH_SIZE}")

texts = df_out["body_clean"].astype(str).tolist()
docs = list(tqdm(nlp.pipe(texts, batch_size=BATCH_SIZE, n_process=N_PROC),
                 total=len(texts), desc="spaCy (no NER)"))

df_out["tokens"]       = [[t.text   for t in d] for d in docs]
df_out["lemmas"]       = [[t.lemma_ for t in d] for d in docs]
df_out["noun_phrases"] = [[np.text  for np in d.noun_chunks] for d in docs]





Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Using paths
Base      : /content/drive/MyDrive/PropInsight
Input     : /content/drive/MyDrive/PropInsight/raw/common_reddit/reddit_corpus_2023_2025.csv
Output    : /content/drive/MyDrive/PropInsight/raw/common_reddit/reddit_corpus_enhanced_common.csv
Report    : /content/drive/MyDrive/PropInsight/raw/common_reddit/reddit_preprocessing_report_common.json
NLP out   : /content/drive/MyDrive/PropInsight/raw/common_reddit/reddit_corpus_enhanced+nlp_common.parquet
Singlish  : /content/drive/MyDrive/PropInsight/corpus/Singlish
Property  : /content/drive/MyDrive/PropInsight/corpus/SGPropertyDomain
Ruler     : /content/drive/MyDrive/PropInsight/corpus/SGPropertyDomain/spacy_entityruler_patterns.merged.jsonl
📘 Singlish terms loaded: 1658 (from 3558 entries)
📗 Property terms loaded from glossary.csv: 1852
📦 Loaded 61918 rows
🧹 Filtered spam/dupes: 0 rows removed; 6191

spaCy (no NER):   0%|          | 0/61918 [00:00<?, ?it/s]

In [ ]:

import spacy
MODEL = "en_core_web_sm"

# ---------------- Optional: NER second pass (kept separate for speed) ----------------
WANT_ENTITIES = True  # <- set True if you also want entities
if WANT_ENTITIES:
    ner_nlp = spacy.load(MODEL, disable=["tagger","parser","attribute_ruler","lemmatizer"])
    if RULER_PATH and Path(RULER_PATH).exists():
        print("🧩 Attaching EntityRuler:", RULER_PATH)
        ruler = ner_nlp.add_pipe("entity_ruler", before="ner")
        ruler.from_disk(str(RULER_PATH))
    ner_docs = list(tqdm(ner_nlp.pipe(texts, batch_size=256, n_process=N_PROC),
                         total=len(texts), desc="spaCy (NER only)"))
    df_out["entities"] = [[{"text": e.text, "label": e.label_} for e in d.ents] for d in ner_docs]
else:
    df_out["entities"] = [[] for _ in range(len(df_out))]

# Save NLP-enriched parquet
df_out.to_parquet(NLP_OUT, index=False)
print("✅ Saved NLP-enriched corpus →", NLP_OUT)

# Quick peek
for i in range(min(3, len(df_out))):
    print("\nTEXT:", df_out.loc[i, "body_clean"][:200])
    print("SING:", df_out.loc[i, "singlish_terms"])
    print("PROP:", df_out.loc[i, "property_terms"])
    print("#TOK:", len(df_out.loc[i, "tokens"]), " #NP:", len(df_out.loc[i, "noun_phrases"]))

NameError: name 'spacy' is not defined